In [ ]:
#Conectare la Google Drive
from google.colab import drive
drive.mount('/content/drive')

#Descarcarea codului sursa
%cd /content
!git clone https://github.com/lucastabelini/LaneATT.git
%cd /content/LaneATT

#Instalarea pachetelor cu versiunile corecte
!pip install -q Shapely xmljson thop imgaug p_tqdm ujson tensorboard gdown kaggle
!pip install -q "numpy>=2.1.0"
!pip install -q "opencv-python-headless<5.0"

#Repararea si compilarea modulului C++
import os
nms_cpp = "/content/LaneATT/lib/nms/src/nms.cpp"
if os.path.exists(nms_cpp):
    with open(nms_cpp, "r") as f: code = f.read()
    code = code.replace("x.type().is_cuda()", "x.is_cuda()")
    with open(nms_cpp, "w") as f: f.write(code)

nms_cu = "/content/LaneATT/lib/nms/src/nms_kernel.cu"
if os.path.exists(nms_cu):
    with open(nms_cu, "r") as f: code = f.read()
    code = code.replace("boxes.type()", "boxes.scalar_type()")
    with open(nms_cu, "w") as f: f.write(code)

%cd lib/nms
!python setup.py install
%cd ../..

#Repararea erorilor din codul original (Splines si Salvarea imaginilor)
dataset_file = "/content/LaneATT/lib/datasets/lane_dataset.py"
with open(dataset_file, "r") as f: code = f.read()
code = code.replace("lanes.append(Lane(points=points))", "if len(points) >= 2: lanes.append(Lane(points=points))")
with open(dataset_file, "w") as f: f.write(code)

runner_file = "/content/LaneATT/lib/runner.py"
with open(runner_file, 'r') as f: lines = f.readlines()
with open(runner_file, 'w') as f:
    for line in lines:
        if "cv2.imshow" in line:
            indent = line.split("cv2")[0]
            f.write(indent + "import uuid\n")
            f.write(indent + "os.makedirs('/content/LaneATT/output_images', exist_ok=True)\n")
            f.write(indent + "cv2.imwrite(f'/content/LaneATT/output_images/pred_{uuid.uuid4().hex[:8]}.jpg', img)\n")
        elif "cv2.waitKey" in line: continue
        else: f.write(line)

#Aducerea modelelor pre-antrenate
!gdown "https://drive.google.com/uc?id=1R638ou1AMncTCRvrkQY6I-11CPwZy23T"
!unzip -q laneatt_experiments.zip -d /content/LaneATT/

#Extragerea instantanee a datelor din Drive
!mkdir -p /content/LaneATT/datasets/
!cp /content/drive/MyDrive/tusimple_ready.zip /content/LaneATT/datasets/
%cd /content/LaneATT/datasets/
!unzip -q tusimple_ready.zip
!ln -s tusimple tusimple-test
print("Mediul este complet pregatit!")

In [ ]:
%cd /content/LaneATT
!python main.py test --exp_name laneatt_r18_tusimple --view all

In [ ]:
#Trimitem pozele cu benzile desenate inapoi in Google Drive-ul tau
%cd /content/LaneATT
!zip -r -q laneatt_results.zip output_images/
!cp laneatt_results.zip /content/drive/MyDrive/
print("Imaginile au fost salvate cu succes in Drive!")